# Email Phishing Classifier — Data Generation

Phase 1 of the plan in `../../PLAN.md`: build a clean, deduplicated, leakage-safe
binary dataset (phishing/malicious vs. legitimate) using Hugging Face datasets only.

**Sources (verified 2026-08-14):**
- `puyang2025/seven-phishing-email-datasets` — 203,017 rows spanning SpamAssassin,
  CEAS-08, Enron, Ling-Spam, and TREC-05/06/07.
- `simlab-vs/meajor_cleaned_preprocessed` — 108,685 anonymized rows spanning
  TREC-05/06/07, Nazario Phishing Corpus, and Nigerian Fraud (CC-BY-4.0).

Both datasets are normalized and merged before cross-source deduplication. The overlapping
TREC corpora make exact and near-duplicate removal mandatory.

**Safety note:** these corpora contain malicious URLs. Never fetch links or open
attachments referenced in the data.

**Outputs:** `data/source-clean/{train,val,test}.parquet`, `adversarial_test.parquet`,
`manifest.json`.

## 1. Configuration

In [1]:
import hashlib, json, os
from pathlib import Path

SEED = 42
PROJECT_DIR = Path(os.environ.get("EMAIL_NN_DIR", "/home/spark/projects/training/email-classifier-nn"))
RAW_DIR = PROJECT_DIR / "data" / "source-raw"
CLEAN_DIR = PROJECT_DIR / "data" / "source-clean"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

HF_SEVEN = "puyang2025/seven-phishing-email-datasets"
HF_MEAJOR = "simlab-vs/meajor_cleaned_preprocessed"
HF_DATASETS = (HF_SEVEN, HF_MEAJOR)

MIN_BODY_CHARS = 20           # quality gate: minimum stripped body length
NEAR_DUP_THRESHOLD = 0.90     # MinHash LSH Jaccard threshold
MINHASH_PERMS = 128
SHINGLE_SIZE = 5              # word shingles for near-dup detection
SPLITS = {"train": 0.80, "val": 0.10, "test": 0.10}
AUG_FRACTION = 0.15           # fraction of train phishing rows to obfuscate

CONFIG_FINGERPRINT = hashlib.sha256(json.dumps({
    "datasets": HF_DATASETS, "min_body": MIN_BODY_CHARS,
    "near_dup": NEAR_DUP_THRESHOLD, "perms": MINHASH_PERMS, "shingle": SHINGLE_SIZE,
    "splits": SPLITS, "aug": AUG_FRACTION, "seed": SEED,
}, sort_keys=True).encode()).hexdigest()
print("config fingerprint:", CONFIG_FINGERPRINT[:16])

config fingerprint: a1e06b36c0af0662


## 2. Environment

In [2]:
%pip install -q datasets pandas pyarrow datasketch

import gc, re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import load_dataset
from datasketch import MinHash, MinHashLSH

URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
WS_RE = re.compile(r"\s+")

def normalize_for_fp(subject, body):
    s = WS_RE.sub(" ", (subject or "").lower()).strip()
    b = WS_RE.sub(" ", (body or "").lower()).strip()
    return s + "\n" + b

def fingerprint(subject, body):
    return hashlib.sha256(normalize_for_fp(subject, body).encode("utf-8", "replace")).hexdigest()

def sender_domain(sender):
    if not isinstance(sender, str):
        return None
    m = re.search(r"@([A-Za-z0-9.-]+)", sender)
    return m.group(1).lower().rstrip(".") if m else None


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/home/spark/projects/data_cleaning_generic/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Download Hugging Face Datasets

Both repositories are loaded with `datasets.load_dataset` and cached as parquet in
`data/source-raw/`; re-runs read the local parquet cache. No external corpus host or
arbitrary URL is contacted by this notebook.

In [3]:
def load_hf_cache(dataset_id, cache_path):
    if cache_path.exists():
        frame = pd.read_parquet(cache_path)
        print(f"{dataset_id} (cached): {len(frame):,} rows")
        return frame

    dataset = load_dataset(dataset_id, split="train")
    frame = dataset.to_pandas()
    tmp = Path(str(cache_path) + ".tmp")
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, cache_path)
    del dataset
    gc.collect()
    print(f"{dataset_id} (downloaded): {len(frame):,} rows")
    return frame


df_seven_raw = load_hf_cache(HF_SEVEN, RAW_DIR / "seven_phishing_raw.parquet")
df_meajor_raw = load_hf_cache(HF_MEAJOR, RAW_DIR / "meajor_raw.parquet")
print(HF_SEVEN, "columns:", df_seven_raw.columns.tolist())
print(HF_MEAJOR, "columns:", df_meajor_raw.columns.tolist())

Generating test split: 100%|██████████| 40604/40604 [00:00<00:00, 113668.11 examples/s]


puyang2025/seven-phishing-email-datasets (downloaded): 162,413 rows


Generating train split: 100%|██████████| 108685/108685 [00:00<00:00, 239703.06 examples/s]


simlab-vs/meajor_cleaned_preprocessed (downloaded): 108,685 rows
puyang2025/seven-phishing-email-datasets columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']
simlab-vs/meajor_cleaned_preprocessed columns: ['sender', 'sender_domain', 'receiver', 'receiver_domain', 'date', 'subject', 'content_types', 'body', 'urls', 'url_count', 'url_length_max', 'url_length_avg', 'url_subdom_max', 'url_subdom_avg', 'attachment_count', 'has_attachments', 'attachment_types', 'language', 'source', 'label']


## 4. Normalize and Merge

Both HF schemas are mapped to a shared representation. `hf_dataset` preserves repository
provenance and `source` preserves the original corpus family. MeAJOR's anonymized body,
URL features, and attachment metadata are retained.

In [4]:
COMMON_COLS = [
    "body", "subject", "sender", "receiver", "date", "urls", "url_count",
    "attachment_count", "has_attachments", "source", "label", "hf_dataset",
]

seven = df_seven_raw.rename(
    columns={"text": "body", "urls": "url_count", "dataset_name": "source"}
).copy()
seven["urls"] = ""
seven["attachment_count"] = np.nan
seven["has_attachments"] = False
seven["hf_dataset"] = HF_SEVEN

meajor = df_meajor_raw.copy()
meajor["hf_dataset"] = HF_MEAJOR

for frame in (seven, meajor):
    missing = set(COMMON_COLS) - set(frame.columns)
    assert not missing, f"missing required columns: {sorted(missing)}"

df = pd.concat([seven[COMMON_COLS], meajor[COMMON_COLS]], ignore_index=True)
df["body"] = df["body"].fillna("").astype(str)
df["subject"] = df["subject"].fillna("").astype(str)
df["label"] = pd.to_numeric(df["label"], errors="coerce")

print(df.groupby(["hf_dataset", "source"], dropna=False).size())
print(df.groupby(["hf_dataset", "label"], dropna=False).size())
print(f"merged rows before quality gates: {len(df):,}")

hf_dataset                                source  
puyang2025/seven-phishing-email-datasets  Assassin     4585
                                          CEAS-08     31150
                                          Enron       23849
                                          Ling         2291
                                          TREC-05     44461
                                          TREC-06     13113
                                          TREC-07     42964
simlab-vs/meajor_cleaned_preprocessed     trec5       49583
                                          trec6       15005
                                          trec7       44096
                                          NaN             1
dtype: int64
hf_dataset                                label
puyang2025/seven-phishing-email-datasets  0.0      86951
                                          1.0      75462
simlab-vs/meajor_cleaned_preprocessed     0.0      60650
                                          1.0      48034


## 5. Quality Gates

Drop rows with invalid labels or near-empty bodies.

In [5]:
n0 = len(df)
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)
df = df[df["body"].str.strip().str.len() >= MIN_BODY_CHARS].copy()
df = df.reset_index(drop=True)
print(f"quality gates: {n0:,} -> {len(df):,} rows ({n0 - len(df):,} dropped)")

quality gates: 271,098 -> 270,039 rows (1,059 dropped)


## 6. Deduplication

1. **Exact**: SHA-256 fingerprint of normalized `subject + body`.
2. **Near-dup**: MinHash LSH over word 5-gram shingles, clustered with union-find.
   Quoted replies/forwards in these corpora create heavy near-duplication; clusters are
   later used as split-grouping keys so near-duplicates never span splits.

Note: building ~200k MinHash signatures is CPU-bound and can take a while.

In [6]:
df["fp"] = [fingerprint(s, b) for s, b in zip(df["subject"], df["body"])]
n0 = len(df)
df = df.drop_duplicates(subset="fp", keep="first").reset_index(drop=True)
print(f"exact dedup: {n0:,} -> {len(df):,}")

def word_shingles(subject, body, k=SHINGLE_SIZE):
    words = normalize_for_fp(subject, body).split()
    if len(words) < k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i + k]) for i in range(len(words) - k + 1)}

parent = list(range(len(df)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

lsh = MinHashLSH(threshold=NEAR_DUP_THRESHOLD, num_perm=MINHASH_PERMS)
signatures = []
for i, (s, b) in enumerate(zip(df["subject"], df["body"])):
    mh = MinHash(num_perm=MINHASH_PERMS)
    for sh in word_shingles(s, b):
        mh.update(sh.encode("utf-8", "replace"))
    signatures.append(mh)
    lsh.insert(str(i), mh)
    if (i + 1) % 20000 == 0:
        print(f"  minhash {i + 1:,}/{len(df):,}")

for i, mh in enumerate(signatures):
    for key in lsh.query(mh):
        j = int(key)
        if j != i:
            union(i, j)

df["dup_cluster"] = [find(i) for i in range(len(df))]
print(f"near-dup clusters: {df['dup_cluster'].nunique():,} for {len(df):,} rows")
del signatures, lsh
gc.collect()

exact dedup: 270,039 -> 264,142
  minhash 20,000/264,142
  minhash 40,000/264,142
  minhash 60,000/264,142
  minhash 80,000/264,142
  minhash 100,000/264,142
  minhash 120,000/264,142
  minhash 140,000/264,142
  minhash 160,000/264,142
  minhash 180,000/264,142
  minhash 200,000/264,142
  minhash 220,000/264,142
  minhash 240,000/264,142
  minhash 260,000/264,142
near-dup clusters: 252,224 for 264,142 rows


1556869

## 7. Leakage-Safe Splits

Grouping key = union of near-dup cluster and sender domain (rows sharing either are
inseparable). Groups are assigned greedily (largest first, seeded shuffle among ties)
to hit the 80/10/10 row targets. No group spans splits by construction.

In [7]:
df["domain"] = df["sender"].map(sender_domain)

domain_first = {}
for i, dom in enumerate(df["domain"]):
    if isinstance(dom, str) and dom:
        if dom in domain_first:
            union(i, domain_first[dom])
        else:
            domain_first[dom] = i

df["group"] = [find(i) for i in range(len(df))]

groups = df.groupby("group").size().rename("n").reset_index()
groups = groups.sample(frac=1.0, random_state=SEED).sort_values("n", ascending=False, kind="stable")

targets = {k: v * len(df) for k, v in SPLITS.items()}
alloc = {k: 0 for k in SPLITS}
assign = {}
for _, g in groups.iterrows():
    split = max(SPLITS, key=lambda k: targets[k] - alloc[k])
    assign[g["group"]] = split
    alloc[split] += g["n"]

df["split"] = df["group"].map(assign)
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))
print(df.groupby(["split", "source"]).size().unstack(fill_value=0))

label       0      1
split               
test    12552  13862
train  118571  92743
val     12575  13839
source  Assassin  CEAS-08  Enron  Ling  TREC-05  TREC-06  TREC-07  trec5  \
split                                                                      
test          96     1445   4062   399     1008      397     1897   7526   
train       4408    28142  15313  1483    42020    12298    38870  30505   
val           74     1423   4148   408      949      363     1953   7630   

source  trec6  trec7  
split                 
test     2311   7273  
train    9704  28571  
val      2361   7105  


## 8. Feature Engineering

Per-email engineered features for the Stage-A baseline (XGBoost / small MLP):
URL stats, lexical/character statistics, obfuscation indicators.

In [8]:
SUSPICIOUS_WORDS = [
    "verify", "account", "password", "urgent", "suspend", "click",
    "confirm", "bank", "invoice", "payment", "login", "update",
]
IP_URL_RE = re.compile(r"https?://\d{1,3}(?:\.\d{1,3}){3}", re.IGNORECASE)
HTML_TAG_RE = re.compile(r"</?[a-zA-Z][^>]*>")
ZW_CHARS = "\u200b\u200c\u200d\ufeff"

def engineer(row):
    body, subject = row["body"], row["subject"]
    urls = URL_RE.findall(body)
    low = (subject + " " + body).lower()
    blen = max(len(body), 1)
    return {
        "f_body_len": len(body),
        "f_subject_len": len(subject),
        "f_url_count": len(urls),
        "f_url_ip": sum(bool(IP_URL_RE.match(u)) for u in urls),
        "f_url_at": sum("@" in u for u in urls),
        "f_upper_ratio": sum(c.isupper() for c in body) / blen,
        "f_digit_ratio": sum(c.isdigit() for c in body) / blen,
        "f_exclaim": body.count("!"),
        "f_html_tags": len(HTML_TAG_RE.findall(body)),
        "f_susp_words": sum(w in low for w in SUSPICIOUS_WORDS),
        "f_nonascii_ratio": sum(ord(c) > 127 for c in body) / blen,
        "f_zero_width": sum(c in ZW_CHARS for c in body),
    }

features = pd.DataFrame([engineer(r) for _, r in df.iterrows()])
FEATURE_COLS = features.columns.tolist()
df = pd.concat([df.reset_index(drop=True), features], axis=1)
print(df[FEATURE_COLS].describe().T[["mean", "std", "max"]])

                         mean          std           max
f_body_len        1555.570273  8670.409811  2.550619e+06
f_subject_len       38.441429    38.479594  7.170000e+03
f_url_count          0.921489     8.422410  3.134000e+03
f_url_ip             0.002135     0.094218  1.800000e+01
f_url_at             0.022484     0.247876  2.700000e+01
f_upper_ratio        0.099145     0.112161  9.859155e-01
f_digit_ratio        0.022554     0.037862  8.518519e-01
f_exclaim            1.308497     7.767209  1.879000e+03
f_html_tags          0.122907     2.897339  4.660000e+02
f_susp_words         0.388568     0.788833  1.200000e+01
f_nonascii_ratio     0.004545     0.047179  1.000000e+00
f_zero_width         0.000008     0.002752  1.000000e+00


## 9. Adversarial Augmentation (train only)

Deterministic obfuscation copies of a seeded sample of **train** phishing rows:
Cyrillic homoglyph substitution + zero-width character injection into trigger words.
A separate fixed adversarial eval set is built from **test** phishing rows (originals
stay in test; the obfuscated copies are saved to `adversarial_test.parquet` only).
Engineered features are recomputed after obfuscation.

In [9]:
HOMOGLYPHS = {
    "a": "\u0430", "c": "\u0441", "e": "\u0435", "i": "\u0456",
    "o": "\u043e", "p": "\u0440", "s": "\u0455", "x": "\u0445",
}
ZW = "\u200b"
TRIGGER_RE = re.compile(
    r"verify|account|password|urgent|bank|login|click|confirm", re.IGNORECASE)

def obfuscate(text, seed):
    r = np.random.default_rng(seed)
    out = []
    for ch in text:
        sub = HOMOGLYPHS.get(ch.lower())
        if sub and r.random() < 0.30:
            out.append(sub.upper() if ch.isupper() else sub)
        else:
            out.append(ch)
    s = "".join(out)
    return TRIGGER_RE.sub(lambda m: m.group(0)[:2] + ZW + m.group(0)[2:], s)

df["augmented"] = False

def make_adversarial(rows, seed_base):
    adv = rows.copy()
    adv["body"] = [obfuscate(b, seed_base + i) for i, b in enumerate(adv["body"])]
    adv["subject"] = [obfuscate(s, seed_base + 500000 + i) for i, s in enumerate(adv["subject"])]
    adv["augmented"] = True
    refreshed = pd.DataFrame([engineer(r) for _, r in adv.iterrows()], index=adv.index)
    adv[FEATURE_COLS] = refreshed
    return adv

train_phish = df[(df["split"] == "train") & (df["label"] == 1)]
train_aug = make_adversarial(train_phish.sample(frac=AUG_FRACTION, random_state=SEED), SEED)

test_phish = df[(df["split"] == "test") & (df["label"] == 1)]
adv_eval = make_adversarial(test_phish, SEED + 1000000)

print(f"train augmentations: {len(train_aug):,}")
print(f"adversarial eval set: {len(adv_eval):,}")

train augmentations: 13,911
adversarial eval set: 13,862


## 10. Verify & Save

Leakage assertions (no shared group or fingerprint across splits), class/source
distribution report, atomic parquet writes, and a fingerprinted manifest.

In [12]:
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ga = set(df.loc[df["split"] == a, "group"])
    gb = set(df.loc[df["split"] == b, "group"])
    assert not (ga & gb), f"group leakage between {a} and {b}"
    fa = set(df.loc[df["split"] == a, "fp"])
    fb = set(df.loc[df["split"] == b, "fp"])
    assert not (fa & fb), f"fingerprint leakage between {a} and {b}"
print("leakage checks passed")

INTERNAL_COLS = ["fp", "dup_cluster", "group", "domain", "split"]

def save_atomic(frame, path):
    tmp = Path(str(path) + ".tmp")
    serializable = frame.drop(columns=INTERNAL_COLS, errors="ignore").copy()
    serializable["date"] = serializable["date"].astype("string")
    serializable.to_parquet(tmp, index=False)
    os.replace(tmp, path)

outputs = {
    "train": pd.concat([df[df["split"] == "train"], train_aug], ignore_index=True)
               .sample(frac=1.0, random_state=SEED).reset_index(drop=True),
    "val": df[df["split"] == "val"].reset_index(drop=True),
    "test": df[df["split"] == "test"].reset_index(drop=True),
    "adversarial_test": adv_eval.reset_index(drop=True),
}
counts = {}
for name, frame in outputs.items():
    path = CLEAN_DIR / f"{name}.parquet"
    save_atomic(frame, path)
    counts[name] = {
        "rows": int(len(frame)),
        "phish": int(frame["label"].sum()),
        "legit": int((frame["label"] == 0).sum()),
        "augmented": int(frame["augmented"].sum()),
    }
    print(f"{name}: {counts[name]}")

manifest = {
    "config_fingerprint": CONFIG_FINGERPRINT,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "sources": list(HF_DATASETS),
    "feature_columns": FEATURE_COLS,
    "counts": counts,
}
tmp = CLEAN_DIR / "manifest.json.tmp"
tmp.write_text(json.dumps(manifest, indent=2))
os.replace(tmp, CLEAN_DIR / "manifest.json")
print("manifest written:", CLEAN_DIR / "manifest.json")

leakage checks passed
train: {'rows': 225225, 'phish': 106654, 'legit': 118571, 'augmented': 13911}
val: {'rows': 26414, 'phish': 13839, 'legit': 12575, 'augmented': 0}
test: {'rows': 26414, 'phish': 13862, 'legit': 12552, 'augmented': 0}
adversarial_test: {'rows': 13862, 'phish': 13862, 'legit': 0, 'augmented': 13862}
manifest written: /home/spark/projects/training/email-classifier-nn/data/source-clean/manifest.json
